In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QESEM: Uma Qiskit Function da Qedma
> **Note:** As Qiskit Functions são um recurso experimental disponível apenas para usuários dos planos IBM Quantum&reg; Premium, Flex e On-Prem (via IBM Quantum Platform API). Elas estão em status de lançamento em pré-visualização e sujeitas a alterações.

## Visão geral
Embora as unidades de processamento quântico tenham melhorado enormemente nos últimos anos, os erros causados por ruído e imperfeições no hardware existente continuam sendo um desafio central para os desenvolvedores de algoritmos quânticos. À medida que a área se aproxima de computações quânticas em escala utilitária que não podem ser verificadas classicamente, soluções para cancelar ruído com precisão garantida tornam-se cada vez mais importantes. Para superar esse desafio, a Qedma desenvolveu o Quantum Error Suppression and Error Mitigation (QESEM), integrado de forma transparente na IBM Quantum Platform como uma [Qiskit Function](/guides/functions).

Com o QESEM, os usuários podem executar seus circuitos quânticos em QPUs ruidosas para obter resultados altamente precisos e livres de erros, com sobrecargas de tempo de QPU altamente eficientes, próximas dos limites fundamentais. Para isso, o QESEM utiliza um conjunto de métodos proprietários desenvolvidos pela Qedma para caracterização e redução de erros. As técnicas de redução de erros incluem otimização de gates, transpilação ciente de ruído, supressão de erros (ES) e mitigação de erros imparcial (EM). Com essa combinação de métodos baseados em caracterização, os usuários podem obter resultados confiáveis e livres de erros para circuitos quânticos genéricos de grande volume, viabilizando aplicações que não poderiam ser realizadas de outra forma.

Para uma descrição completa dos componentes subjacentes, bem como uma demonstração em escala utilitária, consulte o artigo [Reliable high-accuracy error mitigation for utility-scale quantum circuits.](https://arxiv.org/abs/2508.10997)
## Descrição
Você pode usar a função QESEM da Qedma para estimar e executar facilmente seus circuitos com supressão e mitigação de erros, alcançando volumes maiores de circuitos e maiores precisões. Para usar o QESEM, você fornece um circuito quântico, um conjunto de observáveis a medir, uma precisão estatística alvo para cada observável e uma QPU escolhida. Antes de executar o circuito com a precisão alvo, você pode estimar o tempo de QPU necessário com base em um cálculo analítico que não requer a execução do circuito. Quando estiver satisfeito com a estimativa de tempo de QPU, você pode executar o circuito com o QESEM.

Quando você executa um circuito, o QESEM roda um protocolo de caracterização do dispositivo adaptado ao seu circuito, gerando um modelo de ruído confiável para os erros que ocorrem no circuito. Com base na caracterização, o QESEM primeiro implementa transpilação ciente de ruído para mapear o circuito de entrada em um conjunto de qubits físicos e gates, minimizando o ruído que afeta o observável alvo. Isso inclui os gates nativamente disponíveis (CX/CZ em dispositivos IBM&reg;), bem como gates adicionais otimizados pelo QESEM, formando o conjunto estendido de gates do QESEM. O QESEM então executa um conjunto de circuitos ES e EM baseados em caracterização na QPU e coleta os resultados das medições. Esses resultados são então pós-processados classicamente para fornecer um valor esperado imparcial e uma barra de erro para cada observável, correspondendo à precisão solicitada.

![Visão geral do Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
O QESEM demonstrou fornecer resultados de alta precisão para uma variedade de aplicações quânticas e nos maiores volumes de circuitos alcançáveis hoje. O QESEM oferece os seguintes recursos voltados ao usuário, demonstrados na seção de benchmarks abaixo:
-	**Precisão garantida:** O QESEM gera estimativas imparciais para os valores esperados dos observáveis. Seu método EM é equipado com garantias teóricas que — em conjunto com a caracterização de ponta da Qedma — asseguram que a mitigação converge para a saída do circuito sem ruído, dentro da precisão especificada pelo usuário. Em contraste com muitos métodos heurísticos de EM propensos a erros sistemáticos ou vieses, a precisão garantida do QESEM é essencial para assegurar resultados confiáveis em circuitos quânticos e observáveis genéricos.
-	**Escalabilidade para grandes QPUs:** O tempo de QPU do QESEM depende dos volumes de circuito, mas é independente do número de qubits. A Qedma demonstrou o QESEM nos maiores dispositivos quânticos disponíveis hoje, incluindo os dispositivos IBM Quantum Eagle de 127 qubits e Heron de 133 qubits.
-	**Independente de aplicação:** O QESEM foi demonstrado em uma variedade de aplicações, incluindo simulação de Hamiltoniano, VQE, QAOA e estimativa de amplitude. Os usuários podem inserir qualquer circuito quântico e observável a ser medido, e obter resultados precisos e livres de erros. As únicas limitações são ditadas pelas especificações de hardware e pelo tempo de QPU alocado, que determinam os volumes de circuito acessíveis e as precisões de saída. Em contraste, muitas soluções de redução de erros são específicas para aplicações ou envolvem heurísticas não controladas, tornando-as inaplicáveis para circuitos e aplicações quânticas genéricas.
-  **Conjunto estendido de gates:** O QESEM suporta gates de ângulo fracionário e fornece gates $Rzz(\theta)$ de ângulo fracionário otimizados pela Qedma nos dispositivos IBM Quantum Eagle. Esse conjunto estendido de gates permite uma compilação mais eficiente e viabiliza volumes de circuito maiores em até um fator de 2 em comparação com a compilação padrão CX/CZ.
-	**Observáveis multibase:** O QESEM suporta observáveis de entrada compostos por muitas strings de Pauli não comutativas, como Hamiltonianos genéricos. A escolha das bases de medição e a otimização da alocação de recursos de QPU (shots e circuitos) são então realizadas automaticamente pelo QESEM para minimizar o tempo de QPU necessário para a precisão solicitada. Essa otimização, que leva em conta as fidelidades de hardware e as taxas de execução, permite que você execute circuitos mais profundos e obtenha maior precisão.
## Benchmarks
O QESEM foi testado em uma ampla variedade de casos de uso e aplicações. Os exemplos a seguir podem ajudar você a avaliar quais tipos de cargas de trabalho podem ser executadas com o QESEM.

Uma figura de mérito fundamental para quantificar a dificuldade tanto da mitigação de erros quanto da simulação clássica para um dado circuito e observável é o **volume ativo**: o número de gates CNOT que afetam o observável no circuito. O volume ativo depende da profundidade e largura do circuito, do peso do observável e da estrutura do circuito, que determina o cone de luz do observável. Para mais detalhes, veja a palestra do [IBM Quantum Summit 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). O QESEM oferece valor particularmente alto no regime de alto volume, fornecendo resultados confiáveis para circuitos e observáveis genéricos.

![Volume ativo](../docs/images/guides/qedma-qesem/active_volume.svg)

| Aplicação    | Número de qubits | Dispositivo | Descrição do circuito | Precisão | Tempo total | Uso de runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| Circuito VQE  | 8              | Eagle (r3) | 21 camadas no total, 9 bases de medição, cadeia 1D                    | 98%      | 35 min      | 14 min         |
| Kicked Ising   | 28              | Eagle (r3) | 3 camadas únicas x 3 passos, topologia heavy-hex 2D                      | 97%     | 22 min      | 4 min          |
| Kicked Ising   | 28              | Eagle (r3) | 3 camadas únicas x 8 passos, topologia heavy-hex 2D                     | 97%      | 116 min      | 23 min          |
| Simulação de Hamiltoniano Trotterizado   | 40  | Eagle (r3)            | 2 camadas únicas x 10 passos de Trotter, cadeia 1D                    | 97%      | 3 horas     | 25 min         |
| Simulação de Hamiltoniano Trotterizado   | 119  | Eagle (r3)           | 3 camadas únicas x 9 passos de Trotter, topologia heavy-hex 2D                    | 95%      | 6,5 horas     | 45 min         |
| Kicked Ising  | 136             | Heron (r2) | 3 camadas únicas x 15 passos, topologia heavy-hex 2D                    | 99%      | 52 min             | 9 min           |

A precisão é medida aqui em relação ao valor ideal do observável: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, onde '$\epsilon$' é a precisão absoluta da mitigação (definida pelo usuário), e $\langle O \rangle_{ideal}$ é o observável no circuito sem ruído.
'Uso de runtime' mede o uso do benchmark no modo batch (soma do uso de jobs individuais), enquanto 'tempo total' mede o uso no modo session (tempo de parede do experimento), que inclui tempos adicionais de computação clássica e comunicação. O QESEM está disponível para execução em ambos os modos, para que os usuários possam fazer o melhor uso dos seus recursos disponíveis.

Os circuitos Kicked Ising de 28 qubits simulam o Cristal de Quase-Tempo Discreto estudado por Shinjo et al. (veja [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) e [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) em três loops conectados do ibm_kawasaki. Os parâmetros de circuito usados aqui são $(\theta_x, \theta_z) = (0.9 \pi, 0)$, com um estado inicial ferromagnético $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. O observável medido é o valor absoluto da magnetização $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. O experimento Kicked Ising em escala utilitária foi executado nos 136 melhores qubits do ibm_fez; este benchmark específico foi executado no ângulo de Clifford $(\theta_x, \theta_z) = (\pi, 0)$, no qual o volume ativo cresce lentamente com a profundidade do circuito, o que — em conjunto com as altas fidelidades do dispositivo — permite alta precisão em um curto tempo de execução.

Os circuitos de simulação de Hamiltoniano Trotterizado são para um modelo de Ising de Campo Transverso em ângulos fracionários: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ e $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ respectivamente (veja [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). O circuito em escala utilitária foi executado nos 119 melhores qubits do ibm_brisbane, enquanto o experimento de 40 qubits foi executado na melhor cadeia disponível. A precisão é reportada para a magnetização; resultados de alta precisão também foram obtidos para observáveis de maior peso.

O circuito VQE foi desenvolvido em conjunto com pesquisadores do Center for Quantum Technology and Applications do Deutsches Elektronen-Synchrotron (DESY). O observável alvo aqui era um Hamiltoniano composto por um grande número de strings de Pauli não comutativas, enfatizando o desempenho otimizado do QESEM para observáveis multibase. A mitigação foi aplicada a um ansatz otimizado classicamente; embora esses resultados ainda não tenham sido publicados, resultados da mesma qualidade serão obtidos para circuitos diferentes com propriedades estruturais semelhantes.
## Primeiros passos
Autentique-se usando sua [chave de API da IBM Quantum Platform](http://quantum.cloud.ibm.com/) e selecione a Qiskit Function QESEM como segue. (Este trecho assume que você já [salvou sua conta](/guides/functions#install-qiskit-functions-catalog-client) no seu ambiente local.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

qesem_function = catalog.load("qedma/qesem")

## Exemplo
Para começar, experimente este exemplo básico de estimativa do tempo de QPU necessário para executar o QESEM para um dado `pub`:

In [ ]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [ ]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

O exemplo a seguir executa um job QESEM:

In [ ]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Você pode usar as APIs familiares do Qiskit Serverless para verificar o status da sua carga de trabalho da Qiskit Function ou retornar resultados:

In [ ]:
print(sample_job.status())
result = sample_job.result()

## Parâmetros da função
| Nome |  Tipo | Descrição | Obrigatório | Padrão |  Exemplo |
| -----| ------| ------------| -------- | ------- | -------- |
| `pubs` | [EstimatorPubLike](/guides/primitive-input-output) |Esta é a entrada principal. O `Pub` contém 2 a 4 elementos: um circuito, um ou mais observáveis, 0 ou um único conjunto de valores de parâmetros, e uma precisão opcional. Se a precisão não for especificada, o `default_precision` de `options` será usado|  Sim|  N/A |  `[(circuit, [obs1,obs2,obs3], parameter_values, 0.03)]`  |
| `backend_name`| string|Nome do backend a ser usado |Não | O QESEM obterá o dispositivo menos ocupado reportado pela IBM| `"ibm_fez"`|
| `instance` | string|  O nome de recurso de nuvem (CRN) da instância a ser usada nesse formato |  Não |  N/A | `"CRN"`  |
| `options` | dictionary |Opções de entrada. Veja a seção **Options** para mais detalhes. | Não |  Consulte a seção **Options** para detalhes.    |  `{ default_precision = 0.03, "max_execution_time" = 3600, "transpilation_level" = 0}`  |

### Opções
| Opção |  Escolhas | Descrição | Padrão |
| -----| -----------| -------- | ------- |
| `estimate_time_only` |  `"analytical"`  / `"empirical"` / None  | Quando definido, o job QESEM calculará apenas a estimativa de tempo de QPU. Consulte a descrição a seguir para mais detalhes. Quando definido como None, o circuito será executado com o QESEM| None |
|`default_precision` | 0 < float | Será aplicado aos `pubs` que não têm precisão definida. A precisão indica o erro aceitável nos valores esperados dos observáveis em valor absoluto. Ou seja, o tempo de execução da QPU para mitigação será determinado para fornecer valores de saída para todos os observáveis de interesse que estejam dentro de um intervalo de confiança `1σ` da precisão alvo. Se múltiplos observáveis forem fornecidos, a mitigação será executada até que a precisão alvo seja atingida para cada um dos observáveis de entrada. | 0.02|
|`max_execution_time` | 0 < inteiro < 28.800 (8 horas)| Permite limitar o tempo de QPU, especificado em segundos, a ser usado para todo o processo QESEM. Consulte detalhes adicionais abaixo.| 3.600 (uma hora)|
| `transpilation_level` | 0 / 1 | Veja a descrição abaixo | 1|
| `execution_mode` | `"session"` / `"batch"` | Veja a descrição a seguir | "batch"|

> **Caution:** A estimativa de tempo de QPU muda de um backend para outro. Portanto, ao executar a função QESEM, certifique-se de executá-la no mesmo backend que foi selecionado ao obter a estimativa de tempo de QPU.

> **Note:** O QESEM encerrará sua execução quando atingir a precisão alvo ou quando atingir o `max_execution_time`, o que ocorrer primeiro.

- `estimate_time_only` - Este flag permite que os usuários obtenham uma estimativa do tempo de QPU necessário para executar o circuito com o QESEM.
    - Se definido como `"analytical"`, um limite superior do tempo de QPU é calculado sem consumir nenhum uso de QPU. Esta estimativa tem uma resolução de 30 minutos (por exemplo, 30 minutos, 60 minutos, 90 minutos, e assim por diante). Ela é tipicamente pessimista e só pode ser obtida para observáveis Pauli simples ou somas de Paulis sem suportes que se cruzam (por exemplo, Z0+Z1). É principalmente útil para comparar os níveis de complexidade de diferentes parâmetros fornecidos pelo usuário (circuito, precisão, e assim por diante).
    - Para obter uma estimativa mais precisa do tempo de QPU, defina este flag como `"empirical"`. Embora esta opção exija a execução de um pequeno número de circuitos, ela fornece uma estimativa de tempo de QPU significativamente mais precisa. Esta estimativa tem uma resolução de 5 minutos (por exemplo, 20 minutos, 25 minutos, 30 minutos, e assim por diante). O usuário pode escolher executar a estimativa de tempo empírica no modo batch ou session. Para mais detalhes, consulte a descrição do `execution_mode`. Por exemplo, no modo batch, a estimativa de tempo empírica consumirá menos de 10 minutos de tempo de QPU.

- `max_execution_time`: Permite limitar o tempo de QPU, especificado em segundos, a ser usado para todo o processo QESEM. Como o tempo final de QPU necessário para atingir a precisão alvo é determinado dinamicamente durante o job QESEM, este parâmetro permite limitar o custo do experimento. Se o tempo de QPU determinado dinamicamente for menor que o tempo alocado pelo usuário, este parâmetro não afetará o experimento. O parâmetro `max_execution_time` é particularmente útil nos casos em que a estimativa de tempo analítica fornecida pelo QESEM antes do início do job é excessivamente pessimista e o usuário deseja iniciar um job de mitigação mesmo assim. Após o limite de tempo ser atingido, o QESEM para de enviar novos circuitos. Os circuitos que já foram enviados continuam sendo executados (portanto, o tempo total pode ultrapassar o limite em até 30 minutos), e o usuário recebe os resultados processados dos circuitos que foram executados até aquele ponto. Se você quiser aplicar um limite de tempo de QPU menor que a estimativa de tempo analítica, consulte a Qedma para obter uma estimativa da precisão alcançável dentro do limite de tempo.

- `transpilation_level`: Após um circuito ser enviado ao QESEM, ele prepara automaticamente várias transpilações alternativas do circuito e escolhe a que minimiza o tempo de QPU. Por exemplo, transpilações alternativas podem utilizar gates RZZ fracionários otimizados pela Qedma para reduzir a profundidade do circuito. Naturalmente, todas as transpilações são equivalentes ao circuito de entrada em termos de sua saída ideal. Para exercer mais controle sobre a transpilação do circuito, defina o nível de transpilação em `options`. Enquanto `"transpilation_level": 1` corresponde ao comportamento padrão descrito acima, `"transpilation_level": 0` inclui apenas as modificações mínimas necessárias ao circuito original; por exemplo, a 'layerification' — a organização das operações do circuito em 'camadas' de gates de dois qubits simultâneos. Note que o mapeamento automático de hardware para qubits de alta fidelidade é aplicado em qualquer caso.

| transpilation_level | Descrição |
|:-:|:--|
| `1` | Transpilação QESEM padrão. Prepara várias transpilações alternativas e escolhe a que minimiza o tempo de QPU. As barreiras podem ser modificadas na etapa de layerification. |
| `0` | Transpilação mínima: o circuito mitigado se assemelhará estruturalmente ao circuito de entrada. Os circuitos fornecidos no nível 0 devem corresponder à conectividade do dispositivo e devem ser especificados em termos dos seguintes gates: CX, Rzz(α) e gates padrão de qubit único (U, x, sx, rz, e assim por diante). As barreiras serão respeitadas na etapa de layerification. |

- `execution_mode` - O usuário pode escolher executar o job QESEM em uma [sessão IBM](/guides/execution-modes#session-mode) dedicada ou em múltiplos [batches IBM](/guides/execution-modes#batch-mode):
    -   **Modo Session**: As sessões são mais caras, mas resultam em um tempo menor para obter resultados. Assim que a sessão começa, a QPU é reservada exclusivamente para o job QESEM. O cálculo de uso inclui tanto o tempo gasto na execução da QPU quanto as computações clássicas associadas (realizadas pelo QESEM e pela IBM). A Qiskit Function QESEM cuida da criação e encerramento da sessão automaticamente. Para usuários com acesso ilimitado às QPUs (por exemplo, instalações on-premises), recomenda-se usar o modo session para execução mais rápida do QESEM.
    -   **Modo Batch**: No modo batch, a QPU é liberada durante as computações clássicas, levando a um menor uso de QPU. Como os jobs em batch geralmente se estendem por um período mais longo, há um risco maior de desvios de hardware; o QESEM incorpora medidas para detectar e compensar desvios, mantendo a confiabilidade durante execuções prolongadas.

> **Note:** As operações de barreira são tipicamente usadas para especificar as camadas de gates de dois qubits em circuitos quânticos. No nível 0, o QESEM preserva as camadas especificadas pelas barreiras. No nível 1, as camadas especificadas pelas barreiras são consideradas como uma alternativa de transpilação ao minimizar o tempo de QPU.
### Saídas
A saída de uma função Circuit é um [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), que contém dois campos:

- Um objeto [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult). Ele pode ser indexado diretamente a partir do `PrimitiveResult`.

- Metadados em nível de job.

Cada `PubResult` contém um campo `data` e um campo `metadata`.

- O campo `data` contém pelo menos um array de valores esperados (`PubResult.data.evs`) e um array de erros padrão (`PubResult.data.stds`). Também pode conter mais dados, dependendo das opções utilizadas.

- O campo `metadata` contém metadados em nível de PUB (`PubResult.metadata`).

O trecho de código a seguir descreve como recuperar a estimativa de tempo de QPU (quando `estimate_time_only` está definido):

In [ ]:
print(
    f"The estimated QPU time for this PUB is: \n{time_estimate_result[0].metadata}"
)

O trecho de código a seguir demonstra como recuperar os resultados de mitigação (quando `estimate_time_only` não está definido) e as métricas de execução. Eles contêm dados essenciais que permitem uma compreensão mais profunda de como diferentes parâmetros afetam a execução do QESEM. Também pode ser relevante ao escrever um artigo baseado em sua pesquisa.

In [ ]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: \n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n {results.metadata['total_shots']} / {results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

## Buscar mensagens de erro
Se o status da sua carga de trabalho for ERROR, use job.result() para buscar a mensagem de erro da seguinte forma:

In [ ]:
print(sample_job.result())

PrimitiveResult([PubResult(data=DataBin(), metadata={'time_estimation_sec': 12600})], metadata={})


## Get support

The Qedma support team is here to help! If you encounter any issues or have questions about using the QESEM Qiskit Function, please don't hesitate to reach out. Our knowledgeable and friendly support staff are ready to assist you with any technical concerns or inquiries you may have.

You can email us at support@qedma.com for assistance. Please include as much detail as possible about the issue you're experiencing to help us provide a swift and accurate response. You can also contact your dedicated Qedma POC representative via email or phone.

To help us assist you more efficiently, please provide the following information when you contact us:

- A detailed description of the issue
- The job ID
- Any relevant error messages or codes


We are committed to providing you with prompt and effective support to ensure you have the best possible experience with our Qiskit Function.

We are always looking to improve our product and we welcome your suggestions! If you have ideas on how we can enhance our services or features you'd like to see, please send us your thoughts at support@qedma.com

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem)
- Review [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199.](https://arxiv.org/pdf/2507.01199)
- Review [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997.](https://arxiv.org/pdf/2508.10997)


</Admonition>